In [ ]:
import sys
from pathlib import Path

_root = Path().resolve()
if not (_root / "qc").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
from qc.instance import load_instance
from qc.grover_mixer import enumerate_feasible


_DATA = str(_root / "artifacts/data/all_data.csv")
inst = load_instance(_DATA, start=0, T=3)
feas = enumerate_feasible(inst)

beta = np.pi / 2
K = min(len(feas), 40)
off = -(1.0 - np.exp(-1j * beta)) / len(feas)  # exakter Off-Diagonal-Eintrag (nutzt volles |F|)
U = np.full((K, K), off, dtype=complex)
np.fill_diagonal(U, 1.0 + off)  # Diagonale = 1 − (1−e^{−iβ})/|F|

In [2]:
import plotly.express as px

vmax = np.abs(U.real).max()
fig = px.imshow(
    U.real,
    color_continuous_scale="RdBu_r",
    zmin=-vmax,
    zmax=vmax,
    aspect="equal",
    labels={"color": "Re(U)", "x": "Spalte j", "y": "Zeile i"},
    title=f"Grover-Mixer U (Realteil) - {K}×{K}-Ausschnitt",
)
fig.update_layout(width=650, height=600)
fig.show()

In [3]:
np.unique(np.round(U, 10)).size

2

# GM-QAOA: Aufbau und Ablauf

Die zwei Bausteine werden getrennt konstruiert und dann im QAOA-Schaltkreis abwechselnd angewandt:

```text
 Feasibility-Prädikat (qc/instance.py)            Kosten je Konfiguration z
 Lade/Entlade-XOR, Import/Export-XOR,             direkte Kosten (z-only; einziger Term:
 SoC-One-Hot, Outage-Pinning                      Resiliency-Erlös −225 $ je bedientem
          │                                       Outage-Slot) + max über Benders-Cuts (ab R2)
          │ Oracle über 2^8 Muster je Slot                 │  → zwei Online-Slots ⇒ KEIN Outage
          ▼                                                ▼    ⇒ direkte Kosten ≡ 0 ⇒ H_C in R1 leer
 per-Slot-Mengen: 27 (online) / 18 (outage)        H_C = diag(E(z)),  z ∈ F
          │                                        prägt nur Phasen ein,
          │ kartesisches Produkt entlang g_avail   mischt nichts
          ▼
 F: 27 × 27 = 729 feasible Zustände (beide online)
          │
          ▼
 |F⟩ = Gleichverteilung über F
 U_M(β) = I − (1 − e^{−iβ}) |F⟩⟨F|
 mischt all-to-all innerhalb F,
 koppelt nie nach außen
```

Der Schaltkreis alterniert beide Operatoren $p$-mal, mit festen **Ramp-Winkeln** (annealing-artig: $\gamma$ steigt, $\beta$ fällt — kein Parameter-Training):

$$|\psi\rangle \;=\; \underbrace{U_M(\beta_p)\, e^{-i\gamma_p H_C}}_{\text{Layer } p} \cdots \underbrace{U_M(\beta_1)\, e^{-i\gamma_1 H_C}}_{\text{Layer } 1}\; |F\rangle, \qquad P(z) = |\psi_z|^2$$

Rollenverteilung: $H_C$ **schreibt Kosteninformation in die Phasen**, der Mixer **übersetzt Phasendifferenzen in Betragsdifferenzen** (Spiegelung am Amplituden-Mittelwert). Keiner der beiden kann allein eine Verteilung formen — Phasen ändern keine Wahrscheinlichkeiten, und der Mixer auf phasengleichem Zustand ist nur eine globale Phase.

**In diesem Notebook (zwei Online-Slots)** ist genau das der Aufhänger: In Runde 1 gibt es keinen direkten Kostenterm, $H_C$ ist die Nullmatrix, und die Verteilung bleibt exakt uniform. Erst der **erste Gurobi-Cut** füllt die Diagonale — danach hat $H_C$ Struktur und die Amplifikation setzt ein. Die Plots darunter zeigen zuerst die Bausteine (Oracle, leere $H_C$-Diagonale), dann Runde 1 (uniform), dann wie ein einziger Cut die Kosten „bunt" macht.

In [4]:
from qc.instance import direct_costs, int_to_bits

E_direct = direct_costs(int_to_bits(feas, inst.n_bits), inst)
H_C = np.diag(E_direct)

lo, hi = 74, 94  # dasselbe Fenster wie beim Runde-2-Plot weiter unten ⇒ direkter Vergleich
win = list(range(lo, hi))
fig = px.imshow(
    H_C[lo:hi, lo:hi], x=win, y=win,
    color_continuous_scale=[[0.0, "#104281"], [1.0, "#f0efec"]],
    zmin=-125, zmax=0,  # gleiche Skala wie Runde 2 ⇒ „leer" vs. „bunt" 1:1 vergleichbar
    aspect="equal",
    labels={"color": "H_C[z,z] ($)", "x": "Spalte j", "y": "Zeile i"},
    title="Cost-Hamiltonian H_C Runde 1 (Ausschnitt)",
)
fig.update_layout(width=650, height=600)
fig.show()

## Vom Mixer + Cost-Hamiltonian zur Verteilung

Der Zustand $\psi$ lebt nur auf den 729 feasiblen Zuständen — der Mixer koppelt nie nach außen, infeasible Amplituden bleiben exakt 0. Start ist die Gleichverteilung $|F\rangle$: $\psi_z = 1/\sqrt{729}$ für alle $z$.

Ein QAOA-Layer ([qaoa.py:77-79](qc/qaoa.py#L77-L79)):

1. **Kostenphase** — $\psi_z \leftarrow e^{-i\gamma E(z)}\,\psi_z$, mit $H_C$ diagonal und $E(z)$ = auf $[0,1]$ normierte Kosten. Ändert nur die *Phase*, nie $|\psi_z|$.
2. **Grover-Mixer** — $\psi \leftarrow \psi - (1 - e^{-i\beta})\cdot\mathrm{mean}(\psi)$, d. h. $U_M = I - (1-e^{-i\beta})|F\rangle\langle F|$: eine Spiegelung am Mittelwert aller Amplituden.

Nach $p$ Layern wird gemessen: $P(z) = |\psi_z|^2$.

Der Mechanismus: Schritt 1 allein ändert keine einzige Wahrscheinlichkeit. Erst der Mixer macht aus **Phasendifferenzen Betragsdifferenzen** — Zustände, deren Phase nahe an $\mathrm{mean}(\psi)$ liegt, interferieren konstruktiv, die anderen destruktiv. **Ohne Kostendifferenzen gibt es keine Phasendifferenzen, und der Mixer hat nichts zu verstärken** — genau der Fall in Runde 1 dieses (Outage-freien) Notebooks: $E \equiv 0$, die Verteilung bleibt uniform.

In [5]:
from qc.instance import bit_index, direct_costs, int_to_bits
from qc.qaoa import gm_qaoa

bits = int_to_bits(feas, inst.n_bits)
costs1 = direct_costs(bits, inst)

print("g_avail =", inst.g_avail.tolist(), "| Outage-Slots:", int((inst.g_avail == 0).sum()))
print("Kostenlevel Runde 1:", np.unique(costs1), "-> EIN einziges Level (alle", len(feas), "Zustände gleich)")

g_avail = [1, 1, 1] | Outage-Slots: 0
Kostenlevel Runde 1: [0.] -> EIN einziges Level (alle 19683 Zustände gleich)


### Runde 1 ohne Outage: die Verteilung bleibt uniform

**Woher (k)eine Kostenstruktur kommt:** Die Runde-1-Diagonale enthält nur die *direkten* (z-only) Kosten, und davon gibt es genau einen Term — den **Resiliency-Erlös von −225 \$ pro bedientem Outage-Slot** (`y=1`, siehe `qc/instance.py::direct_costs`). Er ist der einzige Kostenanteil, der allein von einem Bit abhängt und nicht von den kontinuierlichen Leistungswerten. Alles andere (ToU-Energie, Demand Charge, Export-Erlös) hängt von `x` ab und kommt erst ab Runde 2 über Benders-Cuts in $H_C$.

**Beide Slots online** ⇒ das `y`-Bit ist per Feasibility überall auf 0 gepinnt ⇒ der Resiliency-Term entfällt ⇒ alle direkten Kosten sind 0 ⇒ $E \equiv 0$ ⇒ die Kostenphase $e^{-i\gamma E}$ ist die Identität. Der Mixer angewandt auf den uniformen Zustand ergibt nur eine globale Phase ($\psi_z = \mathrm{mean}(\psi)$ für alle $z$). Es gibt nichts zu unterscheiden — die Verteilung bleibt exakt uniform, $P(z) = 1/729$.

Das ist **kein Bug**, sondern der Ausgangspunkt: Runde 1 hat ohne Outage schlicht noch keine Kosteninformation. Die kommt erst mit dem ersten Cut aus dem Gurobi-Subproblem — und macht dann, wie unten zu sehen, aus der flachen Verteilung ein mit den Kosten abfallendes Spektrum. (Der Kontrast *mit* Outage — zwei Kostenlevel schon in Runde 1 — steht in `visualize_outage.ipynb`.)

In [6]:
import plotly.graph_objects as go

probs_r1 = gm_qaoa(costs1, p=6)
print("Runde 1:", np.unique(np.round(probs_r1, 12)), "= 1/", len(feas), "überall (uniform)")
print("max |P - 1/|F||  =", float(np.abs(probs_r1 - 1 / len(feas)).max()), "-> keine Amplifikation")

idx = np.arange(len(feas))
fig = go.Figure()
fig.add_hline(y=1 / len(feas), line_dash="dash", line_color="#898781",
              annotation_text="uniform 1/|F|", annotation_font_color="#52514e")
fig.add_scatter(x=idx, y=probs_r1, mode="markers",
                marker=dict(color="#1baf7a", size=5), name="P(z), Runde 1")
fig.update_layout(
    title="Runde 1 (GM-QAOA p=6): H_C leer ⇒ ein Kostenlevel ⇒ flache, uniforme Verteilung",
    xaxis_title="Index des feasiblen Zustands", yaxis_title="P(z)",
    yaxis=dict(range=[0, 2 / len(feas)]),
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

Runde 1: [5.0805263e-05] = 1/ 19683 überall (uniform)
max |P - 1/|F||  = 6.776263578034403e-21 -> keine Amplifikation


### Runde 1: Ein Sample ziehen und dekodieren

Eine Messung des Quantenregisters kollabiert $\psi$ auf genau **einen** Basiszustand $z$, mit Wahrscheinlichkeit $P(z) = |\psi_z|^2$. Im Simulator ist das ein Zug aus der berechneten Verteilung (`rng.choice(..., p=probs)`). In Runde 1 ist die Verteilung uniform — jedes der 729 feasiblen $z$ ist gleich wahrscheinlich. Wir messen `shots`-mal und nehmen das nach aktuellem Kostenmodell günstigste gesampelte $z$ (**best-of-shots**, `qc/qaoa.py::sample_best`); bei flachen Kosten ist das effektiv ein zufälliges feasibles $z$. Genau dieses $z$ geht als fixierte diskrete Belegung ins Gurobi-Subproblem — und liefert den ersten Cut.

Das gezogene $z$ ist ein 16-Bit-Integer, **8 Bit pro Slot, LSB-first**: Bit $b = 8t + \text{Rollenoffset}$ (siehe `qc/instance.py::ROLES`). `decode(z, inst)` zerlegt es in die Rollenbelegung pro Slot:

| Bit (Slot $t$) | Rolle | Bedeutung |
|---|---|---|
| $8t+0$ | `ch` | Batterie lädt |
| $8t+1$ | `dis` | Batterie entlädt (XOR mit `ch`: höchstens eins) |
| $8t+2$ | `imp` | Import aus dem Netz (online frei; Outage: auf 0 gepinnt) |
| $8t+3$ | `exp` | Export ins Netz (XOR mit `imp`; Outage: auf 0 gepinnt) |
| $8t+4..6$ | `b_low/b_mid/b_high` | SoC-Band, one-hot (genau eins = 1) |
| $8t+7$ | `y` | Last im Outage-Slot bedient → Resiliency-Erlös −225 \$ (online: auf 0 gepinnt) |

In [7]:
from qc.instance import ROLES, decode

ROLE_MEANING = {
    "ch":     "Batterie lädt",
    "dis":    "Batterie entlädt",
    "imp":    "Import aus dem Netz (bei Outage auf 0 gepinnt)",
    "exp":    "Export ins Netz (bei Outage auf 0 gepinnt)",
    "b_low":  "SoC-Band niedrig  ┐",
    "b_mid":  "SoC-Band mittel   ├ one-hot: genau eins = 1",
    "b_high": "SoC-Band hoch     ┘",
    "y":      "Last im Outage-Slot bedient (Resiliency-Erlös −225 $; online auf 0 gepinnt)",
}

# Messung simulieren: 1024 Shots aus der QAOA-Verteilung ziehen,
# dann best-of-shots nach aktuellem Kostenmodell (== qc/qaoa.py::sample_best)
rng = np.random.default_rng()
shots = 1024
shot_idx = rng.choice(len(feas), size=shots, p=probs_r1)
best_idx = shot_idx[np.argmin(costs1[shot_idx])]
z = int(feas[best_idx])

print(f"{shots} Shots -> {np.unique(shot_idx).size} verschiedene Zustände gesehen")
print(f"bestes Sample: z = {z}, Bitstring {z:016b} (links Bit 15, rechts Bit 0), "
      f"Kosten {costs1[best_idx]:.0f} $\n")

for t, slot in enumerate(decode(z, inst)):
    status = "online" if inst.g_avail[t] else "OUTAGE"
    print(f"Slot {t} ({status}) — Bits {8 * t}..{8 * t + 7}:")
    for role in ROLES:
        marker = "x" if slot[role] else " "
        print(f"  [{marker}] Bit {bit_index(t, role):>2}  {role:<7} = {slot[role]}   {ROLE_MEANING[role]}")
    print()

1024 Shots -> 997 verschiedene Zustände gesehen
bestes Sample: z = 1714216, Bitstring 110100010100000101000 (links Bit 15, rechts Bit 0), Kosten 0 $

Slot 0 (online) — Bits 0..7:
  [ ] Bit  0  ch      = 0   Batterie lädt
  [ ] Bit  1  dis     = 0   Batterie entlädt
  [ ] Bit  2  imp     = 0   Import aus dem Netz (bei Outage auf 0 gepinnt)
  [x] Bit  3  exp     = 1   Export ins Netz (bei Outage auf 0 gepinnt)
  [ ] Bit  4  b_low   = 0   SoC-Band niedrig  ┐
  [x] Bit  5  b_mid   = 1   SoC-Band mittel   ├ one-hot: genau eins = 1
  [ ] Bit  6  b_high  = 0   SoC-Band hoch     ┘
  [ ] Bit  7  y       = 0   Last im Outage-Slot bedient (Resiliency-Erlös −225 $; online auf 0 gepinnt)

Slot 1 (online) — Bits 8..15:
  [ ] Bit  8  ch      = 0   Batterie lädt
  [ ] Bit  9  dis     = 0   Batterie entlädt
  [ ] Bit 10  imp     = 0   Import aus dem Netz (bei Outage auf 0 gepinnt)
  [x] Bit 11  exp     = 1   Export ins Netz (bei Outage auf 0 gepinnt)
  [ ] Bit 12  b_low   = 0   SoC-Band niedrig  ┐
  [x

### Runde 2: Der Benders-Cut macht die Kosten kontinuierlich

Jetzt der entscheidende Schritt. Nach Runde 1 fixiert Gurobi das gesampelte `z`, löst das kontinuierliche LP und liefert Schattenpreise $\lambda$. Daraus entsteht der Optimality-Cut

$$q(z) \;\ge\; q_0 + \sum_b \lambda_b \, z_b$$

— affin in den Bits, aber mit **kontinuierlichen** Koeffizienten (Dual-Werte in \$). Die $H_C$-Diagonale der Runde 2 ist: direkte Kosten (hier 0) + punktweises Maximum aller bisherigen Cuts.

Damit hat (fast) jeder Zustand seinen **eigenen** Kostenwert $E(z)$ ⇒ eigene Phase $e^{-i\gamma E(z)}$ ⇒ eigenes Interferenzmuster gegen $\mathrm{mean}(\psi)$ ⇒ eigener $P(z)$-Wert. Aus der in Runde 1 **komplett leeren** Diagonale wird durch einen einzigen Gurobi-Aufruf ein mehrstufiges, mit den Kosten fallendes Spektrum — genau das, was der Benders-Loop braucht: das Sampling konzentriert sich zunehmend auf die günstigen Konfigurationen. Der Plot unten zeigt dasselbe $H_C$-Fenster wie oben (Zeilen 74–94) — vorher einfarbig, jetzt bunt.

In [8]:
# Runde 2 — Schritt 1: das Gurobi-Subproblem lösen.
# Gurobi fixiert das diskrete z̄, löst das kontinuierliche LP und liefert
# Zielwert Q(z̄) + Schattenpreise π. Aus π und der affinen RHS-Map entsteht der
# Optimality-Cut q(z) >= q̄ + w·(z − z̄) (z steckt nur in den RHS).
from subproblem.subproblem import solve_subproblem
from qc.benders import build_sub_instance, optimality_cut

for cand in feas[np.argsort(costs1, kind="stable")]:
    res = solve_subproblem(build_sub_instance(inst, int(cand)))
    if res.feasible:
        z = int(cand)
        idx_z = int(np.where(feas == z)[0][0])
        break

print(f"fixiertes z̄ = {z:0{inst.n_bits}b} | Subproblem feasibel: {res.feasible}")
print(f"Q(z̄) = LP-Optimum der kontinuierlichen Restkosten = {res.q_value:.2f} $\n")

x = res.x
print("kontinuierliche Lösung x* (je Slot):")
print(f"  Import   p_imp = {np.round(x['p_imp'], 2)} kW")
print(f"  Export   p_exp = {np.round(x['p_exp'], 2)} kW")
print(f"  Laden    p_ch  = {np.round(x['p_ch'], 2)} kW")
print(f"  Entladen p_dis = {np.round(x['p_dis'], 2)} kW")
print(f"  SoC            = {np.round(x['soc'], 2)} kWh")
print(f"  Peak (Skalar)  = {x['p_peak']:.2f} kW\n")

# Optimality-Cut aus den Duals; am Anker ist er per Konstruktion tight.
cut = optimality_cut(res, bits[idx_z], inst.n_bits)
vals = cut.evaluate(bits)
print(f"Optimality-Cut: tight am Anker — Cut(z̄) = {vals[idx_z]:.2f} == Q(z̄) = {res.q_value:.2f}")

Restricted license - for non-production use only - expires 2027-11-29
fixiertes z̄ = 001000100010001000100010 | Subproblem feasibel: True
Q(z̄) = LP-Optimum der kontinuierlichen Restkosten = 0.00 $

kontinuierliche Lösung x* (je Slot):
  Import   p_imp = [0. 0. 0.] kW
  Export   p_exp = [0. 0. 0.] kW
  Laden    p_ch  = [0. 0. 0.] kW
  Entladen p_dis = [201.26 198.68 206.4 ] kW
  SoC            = [446.96 394.61 340.22] kWh
  Peak (Skalar)  = 0.00 kW

Optimality-Cut: tight am Anker — Cut(z̄) = 0.00 == Q(z̄) = 0.00


In [ ]:
# Runde 2 — Schritt 2: den Cost-Hamiltonian aufbauen und seine Wirkung zeigen.
costs2 = costs1 + vals  # H_C-Diagonale Runde 2: direkte Kosten (0) + max über Cuts (hier: 1 Cut)
print("Runde 2: verschiedene Kostenwerte:", np.unique(costs2).size,
      "->", np.round(np.unique(costs2), 2))

# (a) Dasselbe H_C-Fenster wie in Runde 1 — vorher LEER, jetzt BUNT
H_C2 = np.diag(costs2)
lo, hi = 74, 94
win = list(range(lo, hi))
fig = px.imshow(
    H_C2[lo:hi, lo:hi], x=win, y=win,
    color_continuous_scale=[[0.0, "#104281"], [1.0, "#f0efec"]],
    zmin=-125, zmax=0,
    aspect="equal",
    labels={"color": "H_C[z,z] ($)", "x": "Spalte j", "y": "Zeile i"},
    title="Cost-Hamiltonian H_C Runde 2 (gleicher Ausschnitt)",
)
fig.update_layout(width=650, height=600)
fig.show()

probs_r2 = gm_qaoa(costs2, p=6)
print("verschiedene P(z)-Werte:", np.unique(np.round(probs_r2, 14)).size,
      "| P(argmin-Kosten) =", round(float(probs_r2[np.argmin(costs2)]), 4),
      "| uniform war", round(1 / len(feas), 5))

idx = np.arange(len(feas))
fig = go.Figure()
fig.add_hline(y=1 / len(feas), line_dash="dash", line_color="#898781",
              annotation_text="uniform 1/|F| (Runde 1)", annotation_font_color="#52514e")
fig.add_scatter(
    x=idx, y=probs_r2, mode="markers",
    marker=dict(size=5, color=costs2, colorscale="Viridis",
                colorbar=dict(title="H_C[z,z] ($)"), showscale=True),
    showlegend=False,
)
fig.update_layout(
    title="Runde 2 (GM-QAOA p=6): H_C strukturiert",
    xaxis_title="Index des feasiblen Zustands", yaxis_title="P(z)",
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

Runde 2: verschiedene Kostenwerte: 10 -> [-187.5 -137.5 -125.   -87.5  -75.   -62.5  -37.5  -25.   -12.5    0. ]


verschiedene P(z)-Werte: 10 | P(argmin-Kosten) = 0.0004 | uniform war 5e-05


## Der komplette Benders-Loop

Jetzt alles zusammen — auf derselben Instanz mit zwei Online-Slots. Der Loop startet also
genau aus der oben gezeigten flachen Runde-1-Verteilung und baut sich die Kostenstruktur
Runde für Runde selbst auf. Pro Runde:

1. **Master (Quantum):** $H_C$-Diagonale = direkte Kosten + punktweises Maximum aller
   bisherigen Optimality-Cuts → GM-QAOA → best-of-shots liefert $\bar z$.
2. **Subproblem (klassisch):** Gurobi löst das LP bei fixem $\bar z$.
   *Feasibel* → Optimality-Cut (Duals) verschärft $H_C$, und $g(\bar z) + Q(\bar z)$
   ist ein Kandidat für die obere Schranke (UB).
   *Infeasibel* → Feasibility-Cut (Farkas) streicht alle States mit demselben
   Unlösbarkeits-Beweis aus dem feasiblen Zustandsvektor — der Grover-Mixer über
   den Rest ist implizit (Gleichverteilung über das gefilterte Array).
3. **Schranken:** LB = exaktes Minimum des Master-Modells über die (verbliebene)
   Enumeration — im PoC gratis. Abbruch, sobald UB − LB ≤ gap_tol.

Die QAOA bleibt dabei ein *heuristischer Sampler* — die Benders-Struktur liefert
die Konvergenz, solange der Sampler das Master-Optimum je Runde findet
(bei 729 States mit 4096 Shots praktisch sicher).

In [30]:
from qc.benders import benders_loop, brute_force_optimum

loop = benders_loop(inst, max_rounds=30, gap_tol=1e-4, shots=4096)
print(f"Terminierung: {loop.termination} nach {len(loop.rounds)} Runden, "
      f"Gap = {loop.gap:.6f}\n")
for r in loop.rounds:
    q = f"{r.q:8.2f}" if r.q is not None else "   —    "
    lb = f"{r.lb:9.2f}" if np.isfinite(r.lb) else "     -inf"
    cut = f"Feasibility-Cut: −{r.n_removed} States" if r.n_removed else "Optimality-Cut"
    print(f"Runde {r.round:2d}: z={r.z:0{inst.n_bits}b} {r.status:<10} Q={q}  "
          f"UB={r.ub:9.2f}  LB={lb}  |F|={r.n_states:3d}  {cut}")

Terminierung: gap nach 20 Runden, Gap = 0.000000

Runde  1: z=001000010010011000101010 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=10935  Feasibility-Cut: −8748 States
Runde  2: z=010010100010010001001010 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=7290  Feasibility-Cut: −3645 States
Runde  3: z=010000100010001000010010 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=5103  Feasibility-Cut: −2187 States
Runde  4: z=001001100010001000011000 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=3321  Feasibility-Cut: −1782 States
Runde  5: z=001010100010101000100000 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=1845  Feasibility-Cut: −1476 States
Runde  6: z=001000100010010000100110 optimal    Q= 2955.35  UB=  2955.35  LB=     -inf  |F|=1845  Optimality-Cut
Runde  7: z=000101100010011000101010 infeasible Q=   —      UB=  2955.35  LB=  -838.40  |F|=1170  Feasibility-Cut: −675 States
Runde  8: z=001001010001001000101010 infeasible Q=   —      UB=  2955

In [28]:
rounds_x = [r.round for r in loop.rounds]
fig = go.Figure()
fig.add_scatter(x=rounds_x, y=[r.ub if np.isfinite(r.ub) else None for r in loop.rounds],
                mode="lines+markers", name="UB (bester gefundener Zielwert)",
                line=dict(color="#2a78d6"))
fig.add_scatter(x=rounds_x, y=[r.lb if np.isfinite(r.lb) else None for r in loop.rounds],
                mode="lines+markers", name="LB (Master-Schranke aus Cuts)",
                line=dict(color="#c9563c"))
fig.add_hline(y=loop.best_value, line_dash="dot", line_color="#898781",
              annotation_text=f"Optimum {loop.best_value:.2f} $",
              annotation_font_color="#52514e")
fig.update_layout(
    title="Benders-Konvergenz: obere und untere Schranke pro Runde",
    xaxis_title="Runde", yaxis_title="Gesamtkosten g(z) + q(z)  ($)",
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

In [21]:
# Wie die Cuts das Sampling umlenken: P(z) je Runde, SELBER AUFBAU wie Runde 1/2 —
# ein Punkt je (verbliebenem) feasiblem Zustand über dem Index, Farbe = Master-Kosten.
# Amplifikation ist nicht monoton (GM-QAOA-Interferenz); der Loop lebt von best-of-shots.
fig = go.Figure()
for k, r in enumerate(loop.rounds):
    fig.add_scatter(
        x=np.arange(len(r.states)), y=r.probs, mode="markers",
        marker=dict(size=5, color=r.costs, colorscale="Viridis",
                    colorbar=dict(title="Kosten ($)"), showscale=True),
        name=f"Runde {r.round}", visible=(k == 0),
    )
steps = [dict(method="update", label=f"Runde {r.round}",
              args=[{"visible": [j == k for j in range(len(loop.rounds))]},
                    {"title": f"Runde {r.round}: |F|={len(r.states)}, "
                              f"{'Feasibility-Cut' if r.n_removed else 'Optimality-Cut'}"}])
         for k, r in enumerate(loop.rounds)]
fig.update_layout(
    sliders=[dict(active=0, steps=steps)],
    title=f"Runde {loop.rounds[0].round}: |F|={len(loop.rounds[0].states)}",
    xaxis_title="Index des (verbliebenen) feasiblen Zustands", yaxis_title="P(z)",
    width=750, height=460, plot_bgcolor="#fcfcfb",
)
fig.show()

In [13]:
# |F| pro Runde: Feasibility-Cuts räumen ganze State-Gruppen ab, Optimality-Cuts nicht.
fig = go.Figure()
fig.add_bar(x=rounds_x, y=[r.n_states for r in loop.rounds],
            marker_color=["#c9563c" if r.n_removed else "#2a78d6" for r in loop.rounds],
            text=[f"−{r.n_removed}" if r.n_removed else "" for r in loop.rounds],
            textposition="outside")
fig.update_layout(
    title="Feasible Set pro Runde — rot: Runde mit Feasibility-Cut (Anzahl entfernt)",
    xaxis_title="Runde", yaxis_title="|F| nach der Runde",
    width=750, height=380, plot_bgcolor="#fcfcfb",
)
fig.show()

In [14]:
import pandas as pd
from qc.instance import ROLES, decode
from classical.classical_solver import build_and_solve

START = 0
window = pd.read_csv(_DATA).iloc[START:START + inst.T].reset_index(drop=True)
m_ref, info_ref, _ = build_and_solve([window], scenario_probs=None, time_limit=None,
                                     mip_gap=0.0, log_file="", quiet=True)
v_milp = float(m_ref.ObjVal)

print(f"Loop (Hybrid):  z={loop.best_z:0{inst.n_bits}b}, Gesamtkosten {loop.best_value:.4f} $")
print(f"Gurobi-MILP:    Gesamtkosten {v_milp:.4f} $   "
      f"({info_ref['n_vars']} Var, {info_ref['n_binary_vars']} binär, "
      f"{info_ref['n_constraints']} Constr)")
print(f"→ {'IDENTISCH' if abs(loop.best_value - v_milp) <= 1e-3 else 'ABWEICHUNG!'}\n")

for t, slot in enumerate(decode(loop.best_z, inst)):
    on = "online" if inst.g_avail[t] else "OUTAGE"
    roles = ", ".join(role for role in ROLES if slot[role]) or "alles aus"
    print(f"Slot {t} ({on}): {roles}")
x = loop.best_x
print(f"\nx*: Import {np.round(x['p_imp'], 1)} kW | Export {np.round(x['p_exp'], 1)} kW | "
      f"Laden {np.round(x['p_ch'], 1)} kW | Entladen {np.round(x['p_dis'], 1)} kW")
print(f"    SoC-Verlauf {np.round(x['soc'], 1)} kWh | Peak {x['p_peak']:.1f} kW")

Loop (Hybrid):  z=001010100010101000101010, Gesamtkosten -1.7957 $
Gurobi-MILP:    Gesamtkosten -1.7957 $   (37 Var, 21 binär, 48 Constr)
→ IDENTISCH

Slot 0 (online): dis, exp, b_mid
Slot 1 (online): dis, exp, b_mid
Slot 2 (online): dis, exp, b_mid

x*: Import [0. 0. 0.] kW | Export [48.7 51.3 43.6] kW | Laden [0. 0. 0.] kW | Entladen [250. 250. 250.] kW
    SoC-Verlauf [434.1 368.2 302.4] kWh | Peak 0.0 kW


In [15]:
# Die finale Gurobi-Lösung nach allen Runden — dieselbe Detailtiefe wie beim
# Runde-2-Anker (Schritt 1), damit man Anker (Q=0) und Optimum direkt vergleicht.
xL = loop.best_x
print(f"Optimum: z* = {loop.best_z:0{inst.n_bits}b} | Gesamtkosten = {loop.best_value:.2f} $\n")

print("diskrete Belegung z* (je Slot):")
for t, slot in enumerate(decode(loop.best_z, inst)):
    roles = ", ".join(role for role in ROLES if slot[role]) or "alles aus"
    print(f"  Slot {t} ({'online' if inst.g_avail[t] else 'OUTAGE'}): {roles}")

print("\nkontinuierliche Lösung x* (je Slot):")
print(f"  Import   p_imp = {np.round(xL['p_imp'], 2)} kW")
print(f"  Export   p_exp = {np.round(xL['p_exp'], 2)} kW")
print(f"  Laden    p_ch  = {np.round(xL['p_ch'], 2)} kW")
print(f"  Entladen p_dis = {np.round(xL['p_dis'], 2)} kW")
print(f"  SoC            = {np.round(xL['soc'], 2)} kWh")
print(f"  Peak (Skalar)  = {xL['p_peak']:.2f} kW")

Optimum: z* = 001010100010101000101010 | Gesamtkosten = -1.80 $

diskrete Belegung z* (je Slot):
  Slot 0 (online): dis, exp, b_mid
  Slot 1 (online): dis, exp, b_mid
  Slot 2 (online): dis, exp, b_mid

kontinuierliche Lösung x* (je Slot):
  Import   p_imp = [0. 0. 0.] kW
  Export   p_exp = [48.74 51.32 43.6 ] kW
  Laden    p_ch  = [0. 0. 0.] kW
  Entladen p_dis = [250. 250. 250.] kW
  SoC            = [434.12 368.24 302.36] kWh
  Peak (Skalar)  = 0.00 kW
